Dependencies
terminal pip install commands

pip install pytorch-forecasting --extra-index-url https://download.pytorch.org/whl/cpu

In [841]:
import numpy as np
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet

In [842]:
class DataService:

    def __init__(self, formatted_data_sets):
        self.formatted_data_sets = formatted_data_sets
        
    def format_data(self, data_frames, new_configuration):
        
        for i, index in enumerate(new_configuration["index_column_keys"]):
            
            #print(i)
            #print(index)
            
            for j, index_value in enumerate(new_configuration["index_column_values"][i]):
                
                #print(j)
                #print(index_value)

                # set data
                
                row = data_frames.loc[[index_value]]
                
                time_series_data = row.get(new_configuration["time_series_column_keys"])
                
                time_series_data = time_series_data.set_axis(["Price"], axis=0)
                
                print(time_series_data)
                
                
                row.drop(columns=row.columns, inplace=True)
                
                row = row.assign(Id=[np.repeat(index_value, len(time_series_data.columns))])
                
                row = row.explode("Id")
                
                row.reset_index(drop=True, inplace=True)
                
                row = row.assign(Date=time_series_data.columns.T)
                
                row = row.assign(Prices=time_series_data.T.to_numpy(copy=True))
                
                print(row)
                
                self.formatted_data_sets[index_value] = row
                
                break

In [843]:
data_frames = pd.read_csv("Metro_new_con_median_sale_price_per_sqft_uc_sfrcondo_month.csv")

data_frames.set_index("RegionID", inplace=True)
data_frames.sort_index(inplace=True)

#time_series_column_range = range(6, 102, 1)

#print(data_frames.get(data_frames.columns[6:len(data_frames.columns)]))

new_configuration = {
    "index_column_keys": ["RegionID"],
    "index_column_values": [data_frames.index[0:1:1]],
    "time_series_column_keys": data_frames.columns[4:len(data_frames.columns)],
    "transpose_time_series_data": True
}

#print(new_configuration)

data_service = DataService({})
data_service.format_data(data_frames, new_configuration)

       2018-01-31  2018-02-28  2018-03-31  2018-04-30  2018-05-31  2018-06-30  \
Price  134.134666  134.826025  136.998984  136.775042  139.170897  140.513529   

       2018-07-31  2018-08-31  2018-09-30  2018-10-31  ...  2025-04-30  \
Price  140.732456  143.780902  145.533981  146.650069  ...  205.191697   

       2025-05-31  2025-06-30  2025-07-31  2025-08-31  2025-09-30  2025-10-31  \
Price  203.626298  203.934783  203.797554  204.751439  204.545455   206.69844   

       2025-11-30  2025-12-31  2026-01-31  
Price   203.47587  201.694232  201.995012  

[1 rows x 97 columns]
        Id        Date      Prices
0   102001  2018-01-31  134.134666
1   102001  2018-02-28  134.826025
2   102001  2018-03-31  136.998984
3   102001  2018-04-30  136.775042
4   102001  2018-05-31  139.170897
..     ...         ...         ...
92  102001  2025-09-30  204.545455
93  102001  2025-10-31  206.698440
94  102001  2025-11-30  203.475870
95  102001  2025-12-31  201.694232
96  102001  2026-01-31  201.9